## 1-minute introduction to Jupyter ##

A Jupyter notebook consists of cells. Each cell contains either text or code.

A text cell will not have any text to the left of the cell. A code cell has `In [ ]:` to the left of the cell.

If the cell contains code, you can edit it. Press <kbd>Enter</kbd> to edit the selected cell. While editing the code, press <kbd>Enter</kbd> to create a new line, or <kbd>Shift</kbd>+<kbd>Enter</kbd> to run the code. If you are not editing the code, select a cell and press <kbd>Ctrl</kbd>+<kbd>Enter</kbd> to run the code.

---

# Object-Oriented Programming

This lesson begins a new chapter on **Object-Oriented Programming**, a different paradigm of programming. Up to this point we have been learning imperative programming, a style of programming that focuses on writing line after line of commands to the computer, repeating operations with loops, selecting operations with branching, and chunking operations with procedures (which we treat as functions).

In lesson 14, despite applying abstraction as best as we can, we are still left with a number of challenges. Let's see how object-oriented programming, a different way of thinking about code organisation, can help address some of them.

## Object-Oriented Programming Principles

Object-Oriented Programming (OOP) is centred around three principles: **Encapsulation**, **Polymorphism**, and **Inheritance**. We will cover one principle in each lesson, beginning with some code, explaining how the principle addresses some challenges, and demonstrating how it is applied to the code.

**Note:** Many websites will mention a fourth principle, Abstraction. This principle is not mentioned in the 9569 syllabus, and here will be treated as a more general principle that applies beyond OOP.

### Starting code

Following on from lesson 14, and applying the improvements discussed, we have our starting code in `starting_code.py`. It is not displayed here so as to keep the lesson document short, so take a pause at this point to have a look first before we proceed.

[Open starting_code.py](starting_code.py)

It's 245 lines of code, seems to be working fine but not thoroughly tested, so don't be surprised to find bugs!

Can we make it even more readable, perhaps shorter, and even maybe find some bugs along the way?

## Encapsulation

One of the challenges mentioned in [Lesson 14](../Programming%20in%20Python/lesson_14.ipynb) is that even with the function parameters labelled and annotated, it's easy to mix them up. We have two kinds of boards here, a targetting board that tracks hits and misses, and a playing board that tracks ship positions.

(_We could combine the two into one board, but didn't because that would make it harder to show the players a targetting board without also showing the opponent's ships._)

For example:

```python
def display_overlay(targetting: list[list[str]], board: list[list[str]]) -> None:
```

Even with this function signature, it's possible, even easy, to accidentally swap the targetting board and playing board. Worse, we might swap the player and enemy boards!

Furthermore, there is information that is closely related to the board, such as the `GRID_SIZE`, and the `player_ship_cells` dict that keeps track of the number of cells remaining for each ship. It would be all too easy to update one of these items and forget to update other items.

We can make it easier to access these items together by **bundling** them into an **object**.

### Bundling data

#### Bundling with dicts

In python, we've seen that dicts can be used to bundle data. For example, we could create a board as follows:

In [ ]:
from starting_code import GRID_SIZE, create_grid
board = {
    "grid": create_grid(GRID_SIZE, "~"),
    "grid_size": GRID_SIZE
}
board

But this still does not solve the problem of accidentally swapping targetting and playing boards.

#### Bundling with objects

We could create an **object**. We can bind variables to an object; such variables are called **attributes**.

To do so, we need to create a **class**. A class is a blueprint for creating objects. In technical terms, we say that:

- a class is used to **instantiate** an object, OR
- an object is an instance of a class

Think of a rubber stamp. We can use it (with an inkpad) to stamp patterns of different colours. But the stamped image is different from the rubber stamp. Likewise, objects are instances of their classes, but are not the same as their class.

Let's create a `Grid` class:

In [ ]:
# the `class` keyword is used to define a class
class Grid:
    pass

board = Grid()
board.grid = create_grid(GRID_SIZE, "~")
board.grid_size = GRID_SIZE

print("board.grid attribute:", board.grid)
print("board.grid_size attribute:", board.grid_size)
grid

Notice that trying to access `grid` and `grid_size` directly doesn't work; they are attributes of `board` and must be accessed from `board` using the `object.attribute` notation (e.g. `board.grid` and `board.grid_size`), not as variables.

It is annoying to have to write 3 lines of code to create a prepared grid though, so we would usually use a _factory function_, which are functions used to create objects:

In [ ]:
def create_board(grid_size: int) -> Grid:
    board = Grid()
    board.grid = create_grid(grid_size, "~")
    board.grid_size = grid_size
    return board

board = create_board(GRID_SIZE)
board

Notice that Python recognises `board` as an instance of `Grid`; we can use `isinstance()` to check this:

In [ ]:
isinstance(board, Grid)

Python offers an even easier way to create a prepared object: instead of having to write a separate factory function (which is also easy to mix up with other factory functions), it lets us **bundle the factory function into the class**. Such bundled factory functions are called **constructor methods**.

Just as an attribute is a variable bound to an object, a **method** is a function bound to an object. Attributes and methods must be accessed through the object, using `object.attribute` and `object.method()` syntax.

In Python, we define a constructor method as follows:

In [ ]:
from starting_code import EMPTY

class Grid:
    """Represents a grid for a game board.
    
    In Python, a class can have a docstring as well. This docstring is added immediately
      after the class definition.
    """
    def __init__(self, grid_size: int, char: str = EMPTY):
        """__init__() is the constructor method for the class.
        This special name is determined by Python and cannot be changed.
        In a Python method definition, the first argument is always `self`, which refers
          to the instance that is being created.
        (It need not be named `self`, but it is a Python convention to do so.)
        To access or set the variables of the object being created, we do so from `self`.
        """
        self.grid = create_grid(grid_size, char)
        self.grid_size = grid_size
        # Note that no return statement is required.
        # Be careful not to write:
        # grid = ...
        # This would create a new local variable `grid` that is not accessible outside
        # the constructor method, instead of setting the instance variable `self.grid`.

# A class with a constructor method can be instantiated like this:
board = Grid(GRID_SIZE)
# Note that we do not need to pass `self` as an argument to the constructor method.
# Python automatically passes the instance as the first argument to the constructor method.
board

### Using objects: `place_ship()`

Let's see what it is like refactoring the code to use an object. **Refactoring** refers to the process of changing the structure of code without changing its behaviour. We will refactor the `place_ship()` function to use the `Grid` class we just created.

#### Before

```python
ef place_ship(board: list[list[str]], ship_name: str, size: int) -> None:
    """Place a ship of size size on the board at a random location."""
    orientation = random.choice([HORIZONTAL, VERTICAL])
    placed = False
    while not placed:
        row = random.randint(0, GRID_SIZE - 1)
        col = random.randint(0, GRID_SIZE - 1)
        if orientation == HORIZONTAL and col + size <= GRID_SIZE:
            if is_unoccupied(board, row, col, size, HORIZONTAL):
                place_ship_horizontally(board, ship_name, size, row, col)
                placed = True
        elif orientation == VERTICAL and row + size <= GRID_SIZE:
            if is_unoccupied(board, row, col, size, VERTICAL):
                place_ship_vertically(board, ship_name, size, row, col)
                placed = True
```

#### After

```python
def place_ship(board: list[list[str]], ship_name: str, size: int) -> None:
    """Place a ship of size size on the board at a random location."""
    orientation = random.choice([HORIZONTAL, VERTICAL])
    placed = False
    while not placed:
        row = random.randint(0, board.grid_size - 1)
        col = random.randint(0, board.grid_size - 1)
        if orientation == HORIZONTAL and col + size <= board.grid_size:
            if is_unoccupied(board, row, col, size, HORIZONTAL):
                place_ship_horizontally(board, ship_name, size, row, col)
                placed = True
        elif orientation == VERTICAL and row + size <= board.grid_size:
            if is_unoccupied(board, row, col, size, VERTICAL):
                place_ship_vertically(board, ship_name, size, row, col)
                placed = True
```

Of course, we would need to refactor other functions to also use the `Grid` class, but we won't do that yet.

... That seems underwhelming; all we did was eliminate the use of `GRID_SIZE` in the function. But encapsulation is not only applied to bundle data together; it also allows us to bundle related functions into the class.

## Bundling methods

For example, `is_unoccupied(board, row, column, size, orientation)` is a function that checks if a ship can be placed at a given location. It is closely related to the `Grid` class, so we can bundle it into the class as a method:

In [ ]:
from starting_code import HORIZONTAL, VERTICAL, EMPTY

class Grid:
    """Represents a grid for a game board."""
    def __init__(self, grid_size: int, char: str = EMPTY):
        """For commonly understood methods like __init__, it is fine to leave out docstrings."""
        self.grid = create_grid(grid_size, char)
        self.grid_size = grid_size

    def is_unoccupied(self, row: int, col: int, size: int, orientation: str) -> bool:
        """Check if the specified area on the board is unoccupied.

        Return True if unoccupied, False otherwise.

        (don't forget to add the `self` parameter when converting a function to the method!)
        (also notice that `self` has replaced the `board` parameter, which is no longer needed)
        """
        if orientation == HORIZONTAL:
            for i in range(size):
                if self.grid[row][col + i] != EMPTY:
                    return False
        elif orientation == VERTICAL:
            for i in range(size):
                if self.grid[row + i][col] != EMPTY:
                    return False
        return True

board = Grid(GRID_SIZE)
board.is_unoccupied(0, 0, 3, HORIZONTAL)

It's nicer to write `board.is_unoccupied(row, col, size, HORIZONTAL)` than `is_unoccupied(board, row, col, size, HORIZONTAL)`, isn't it? Better still, we will never pass the wrong board to the method, because the board isn't even a parameter of the method; it is guaranteed to act on the correct board.

### Exercise: bundling data and methods

Practise applying what you just learnt. Bundle the following methods into the `Grid` class:
- `place_ship_horizontally()`
- `place_ship_vertically()`
- `place_ship()`
- `display_board()` (rename as `display()`)

You may copy the code from the previous cell and continue working from there.

In [ ]:
from starting_code import HORIZONTAL, VERTICAL, EMPTY

# Define and implement your `Grid` class below.



### Separating interface from implementation

Besides bundling, we can also design methods in a way that let us separate the interface from the implementation.

When we use Python functions, such as `random.randint()` or `divmod()`, notice that we don't actually know *how* they work. By reading the documentation, we are told what argument types they expect, what they return, and what they do. And that is enough for us to use them. We don't need to know how they are implemented.

The **interface** is the part of the code that is visible to the user: for us, that's usually the function/method parameter types, the return type, and the docstring.

The **implementation** is the part of the code that is hidden from the user: for us, that is the inner workings of the function/method, i.e. the "code".

A good class design makes it possible to use the class without needing to know how it works. Our `Grid` class does not qualify yet, because many parts of the code rely on its attributes being a specific type. For example, in the `Grid.is_unoccupied()` method, we have the following line:

```python
if self.grid[row][col + i] != EMPTY:
```

Notice this relies on `self.grid` being a list of lists. This seems fine and natural to you at the moment, because you can't imagine other ways of implementing a 2D grid; surely it has to be a list of lists?!

But as you improve in skill, or as you add more functionality to the game such that a list of list of `str`s is no longer sufficient, we'll need to change the grid format. But doing this will require carefully inspecting the code, checking every place where we treated `Board.grid` as a list of lists, and changing it to the new format. **This is error-prone and tedious.**

How would we design the interface of the `Grid` class to improve this? In classic OOP, we treat most attributes as **private**. This means that they are not directly accessible from outside the class. Instead, we provide **getter** and **setter** methods to access and modify them. In programming languages where this is enforced, such as Java, our `Grid` class would look like this:

```python
class JavaGrid:
    def __init__(self, grid_size: int, char: str) -> None:
        self._grid = create_grid(grid_size, char)
        self._grid_size = grid_size
    
    def get_grid(self) -> list[list[str]]:
        return self._grid

    def set_grid(self, grid: list[list[str]]) -> None:
        self._grid = grid
    
    def get_grid_size(self) -> int:
        return self._grid_size

    def set_grid_size(self, grid_size: int) -> None:
        self._grid_size = grid_size
```

### Defining the public interface

In Python, we do things differently. Python does not have a notion of **private attributes**. Instead, attributes are named following conventions to indicate that they are non-public. In other words, users can still access them, but do so at their own risk. Such access is discouraged.

For example, to mark our `Grid.grid` attribute as non-public, we can prefix it with an underscore:

```python
class Grid:
    def __init__(self, grid_size: int, char: str) -> None:
        self._grid = create_grid(grid_size, char)  # marked as non-public
        self.grid_size = grid_size  # public
```

This describes a *contract*: as programmers, we are effectively telling other users "you may access the `Grid.grid_size` attribute directly, but you should not access the `Grid._grid` attribute". This is a convention, and it is up to the user to respect it. If they don't, they are responsible for any bugs that arise from this.

How can users make changes to the grid then? We provide **public methods** for them to do so. Commonly, we will provide **getter** methods for them to get data, and **setter** methods for them to set data, also other utility methods to display data, or retrieve it in a different format.

To summarise:

- we **separate interface from implementation**
- by marking **protected attributes as private** (which means they should not be accessed directly by users)
- and providing **public methods** to access and modify them

### `Grid`'s public interface

Let's define the public interface of our `Grid` class. We will provide the following public methods:

- `Grid.get(x: int, y: int) -> str`: returns the `str` at the given coordinates
- `Grid.set(x: int, y: int, char: str) -> None`: sets the `str` at the given coordinates
- `Grid.is_occupied() -> bool`: returns `True` if the cell is occupied, `False` otherwise (empty)
- `Grid.display() -> None`: displays the formatted grid

And we will also provide the following public attributes:

- `Grid.grid_size`: the square grid's size. Users may access this attribute directly, but should not modify it.

#### Using the public interface

Below, I have rewritten two functions, `is_target_hit()` and `targetting update()` to only use the public interface, without touching `Grid.grid`:

In [ ]:
from starting_code import EMPTY, HIT, MISS

def is_target_hit(board: Grid, x: int, y: int) -> bool:
    """Check if the target hit a ship. Return True if it did, False otherwise."""
    return board.get(x, y) != EMPTY

def targetting_update(board: Grid, hit_what: str, x: int, y: int) -> None:
    """Update the targetting board with the guess."""
    if hit_what == EMPTY:
        board.set(x, y, MISS)
    else:
        board.set(x, y, HIT)

Now we have eliminated any assumption about the grid's internal representation and format. We use the **interface**—the public `get()` and `set()` methods—to update the board data, which lets us use the board without having to know anything about its **implementation**—the `_grid` attribute. This is a good design, because it allows us to change the implementation without breaking the code that uses it.

For example, our `Grid` might look like this now:

In [ ]:
# Docstrings and other methods of the interface are omitted for brevity.
from starting_code import HORIZONTAL, VERTICAL, EMPTY

class Grid:
    """Represents a grid for a game board.
    This implementation uses a 2D list of lists to represent the grid.
    """
    def __init__(self, grid_size: int, char: str = EMPTY):
        self._grid = create_grid(grid_size, char)
        self.grid_size = grid_size

    def get(self, x: int, y: int) -> str:
        return self._grid[x][y]
    
    def set(self, x: int, y: int, char: str) -> None:
        self._grid[x][y] = char

    def is_unoccupied(self, row: int, col: int, size: int, orientation: str) -> bool:
        if orientation == HORIZONTAL:
            for i in range(size):
                if self.get(row, col + i) != EMPTY:
                    return False
        elif orientation == VERTICAL:
            for i in range(size):
                if self.get(row + i, col) != EMPTY:
                    return False
        return True

### Refactoring the implementation painlessly

If I decide to use a different implementation of the grid--such as a 1D list--I can do so without breaking the code that uses it. I just need to change the `get()` and `set()` methods to use the new implementation, and everything else will work as before.

In [ ]:
# Docstrings and other methods of the interface are omitted for brevity.
from starting_code import HORIZONTAL, VERTICAL, EMPTY

class Grid:
    """Represents a grid for a game board.
    This implementation uses a 1D list to represent the grid.
    coordinates (x, y) are translated into an index using the formula `x * grid_size + y`.
    """
    def __init__(self, grid_size: int, char: str = EMPTY):
        self._grid = [char] * (grid_size * grid_size)  ## Use a 1D list to represent 2D grid
        self.grid_size = grid_size

    def get(self, x: int, y: int) -> str:
        return self.grid[x * self.grid_size + y]
    
    def set(self, x: int, y: int, char: str) -> None:
        self.grid[x * self.grid_size + y] = char

    def is_unoccupied(self, row: int, col: int, size: int, orientation: str) -> bool:
        if orientation == HORIZONTAL:
            for i in range(size):
                if self.get(row, col + i) != EMPTY:
                    return False
        elif orientation == VERTICAL:
            for i in range(size):
                if self.get(row + i, col) != EMPTY:
                    return False
        return True
    
# Any code using `Grid` will continue to work without requiring any change,
# as long as they abide by the contract and did not access the grid directly,
# and provided this implementation is free of bugs.

### Exercise: separating interface from implementation

Practise applying what you just learnt. Copying your code from the previous exercise ["Exercise: bundling data and methods"](#exercise-bundling-data-and-methods): separate the interface from the implementation of the `Grid` class. You will need to:
- implement the `get()` method
- implement the `set()` method
- refactor the other methods to use the `get()` and `set()` methods, instead of accessing the `_grid` attribute directly

Test your encapsulated `Grid` class. Does it work as expected?

In [ ]:
# Write your code here

# Summary

Research shows that **active recall**, the mental effort of attempting to remember, helps strengthen neuron connections. For each of the questions below, try to recall what you learnt from this lesson before you click to reveal.

<ol>

<li><details>
    <summary>What is an <b>object</b>? How is it related to a class? (click to reveal)</summary>
    <p>An object bundles related attributes and methods. It is instantiated from a class.</p>
    <p>A class is a blueprint used to instantiate an object.</p>
</details></li>
    
<li><details>
    <summary>What is an <b>attribute</b>? How does it differ from a variable? (click to reveal)</summary>
    <p>An attribute is a variable bound to an object. It must be accessed through the object (using <code>object.attribute</code> syntax).</p>
</details></li>

<li><details>
    <summary>What is a <b>method</b>? How does it differ from a function? (click to reveal)</summary>
    <p>A method is a function bound to an object. It must be accessed through the object (using <code>object.method()</code> syntax).</p>
</details></li>

<li><details>
    <summary>What is <b>encapsulation</b>? (click to reveal)</summary>
    <p>Encapsulation refers to the bundling of related attributes and methods into an object.</p>
</details></li>

<li><details>
    <summary>How does encapsulation prevent <b>inadvertent modification</b> of code? (click to reveal)</summary>
    <p>Encapsulation allows <b>separation of interface from implementation</b>. Protected data is represented as <b>private attributes</b>, which may only be accessed through <b>public methods</b>. The public methods provide safe ways to modify the protected data while maintaining consistency and integrity of data.</p>
</details></li>